# 📈 Enhanced Momentum & Mean-Reversion Strategy
This notebook implements an optimized trading strategy combining momentum and mean-reversion signals with advanced risk management.

In [ ]:
# Setup
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ta
from scipy import stats

In [ ]:
def prepare_data(symbol, start_date, end_date):
    """Download and prepare data with enhanced indicators"""
    df = yf.download(symbol, start=start_date, end=end_date)
    
    # Basic indicators
    df['RSI'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    bb = ta.volatility.BollingerBands(df['Close'])
    df['BB_High'] = bb.bollinger_hband()
    df['BB_Low'] = bb.bollinger_lband()
    df['BB_Mid'] = bb.bollinger_mavg()
    df['BB_Width'] = (df['BB_High'] - df['BB_Low']) / df['BB_Mid']
    
    # Additional indicators
    df['MACD'] = ta.trend.MACD(df['Close']).macd()
    df['MACD_Signal'] = ta.trend.MACD(df['Close']).macd_signal()
    df['ATR'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
    df['ADX'] = ta.trend.ADXIndicator(df['High'], df['Low'], df['Close']).adx()
    
    # Volume indicators
    df['OBV'] = ta.volume.on_balance_volume(df['Close'], df['Volume'])
    df['MFI'] = ta.volume.money_flow_index(df['High'], df['Low'], df['Close'], df['Volume'])
    
    return df.dropna()

In [ ]:
def calculate_signals(df, rsi_thresh=50, bb_dev=2.0, adx_thresh=25):
    """Enhanced signal generation with multiple confirmations"""
    df['Signal'] = 0
    df['Signal_Strength'] = 0
    
    # Calculate signal conditions
    momentum_buy = (df['RSI'] > rsi_thresh) & (df['RSI'].shift(1) <= rsi_thresh)
    momentum_sell = (df['RSI'] < rsi_thresh) & (df['RSI'].shift(1) >= rsi_thresh)
    
    reversion_buy = (
        (df['Close'] < df['BB_Low']) & 
        (df['Close'].shift(1) >= df['BB_Low'].shift(1)) &
        (df['MFI'] < 30)  # Oversold on MFI
    )
    
    reversion_sell = (
        (df['Close'] > df['BB_High']) & 
        (df['Close'].shift(1) <= df['BB_High'].shift(1)) &
        (df['MFI'] > 70)  # Overbought on MFI
    )
    
    # Strong trend confirmation
    strong_trend = df['ADX'] > adx_thresh
    
    # Volume confirmation
    volume_confirm = df['OBV'] > df['OBV'].rolling(20).mean()
    
    # Generate signals with strength
    df.loc[momentum_buy & volume_confirm, 'Signal'] = 1
    df.loc[reversion_buy & volume_confirm, 'Signal'] = 1
    df.loc[momentum_sell | reversion_sell, 'Signal'] = -1
    
    # Calculate signal strength (0 to 1)
    df['Signal_Strength'] = df['Signal'].abs() * (
        (df['ADX'] / 100) * 0.4 +  # Trend strength
        (abs(50 - df['RSI']) / 50) * 0.3 +  # RSI extremity
        (df['BB_Width'] / df['BB_Width'].rolling(20).mean()) * 0.3  # Volatility regime
    )
    
    return df

In [ ]:
def backtest_strategy(df, initial_capital=10000, position_size=0.95, stop_loss=0.02, take_profit=0.03):
    """Enhanced backtest with position sizing and risk management"""
    capital = initial_capital
    position = 0
    entry_price = 0
    df['Portfolio'] = np.nan
    trades = []
    
    for i in range(1, len(df)):
        current_price = df['Close'].iloc[i]
        signal = df['Signal'].iloc[i]
        atr = df['ATR'].iloc[i]
        
        # Dynamic position sizing based on ATR
        risk_per_trade = capital * 0.01  # Risk 1% per trade
        position_size = risk_per_trade / (atr * 2)  # Use 2x ATR for stop loss
        
        # Check stop loss and take profit
        if position != 0:
            pnl_pct = (current_price - entry_price) / entry_price
            
            if (position > 0 and pnl_pct <= -stop_loss) or (position > 0 and pnl_pct >= take_profit):
                # Exit position
                capital = position * current_price
                trades.append({
                    'exit_date': df.index[i],
                    'exit_price': current_price,
                    'pnl': (current_price - entry_price) * position,
                    'pnl_pct': pnl_pct
                })
                position = 0
        
        # Enter new positions
        if signal == 1 and position == 0:
            position = (capital * position_size) / current_price
            entry_price = current_price
            capital = capital - (position * current_price)
            trades.append({
                'entry_date': df.index[i],
                'entry_price': current_price
            })
            
        elif signal == -1 and position > 0:
            capital = position * current_price
            trades.append({
                'exit_date': df.index[i],
                'exit_price': current_price,
                'pnl': (current_price - entry_price) * position,
                'pnl_pct': (current_price - entry_price) / entry_price
            })
            position = 0
            
        df.iloc[i, df.columns.get_loc('Portfolio')] = capital + (position * current_price)
    
    return df, trades

In [ ]:
def calculate_metrics(df, trades):
    """Calculate comprehensive performance metrics"""
    daily_returns = df['Portfolio'].pct_change().dropna()
    
    # Basic metrics
    total_return = (df['Portfolio'].iloc[-1] / df['Portfolio'].iloc[0]) - 1
    sharpe_ratio = daily_returns.mean() / daily_returns.std() * np.sqrt(252)
    sortino_ratio = daily_returns.mean() / daily_returns[daily_returns < 0].std() * np.sqrt(252)
    max_drawdown = (df['Portfolio'] / df['Portfolio'].cummax() - 1).min()
    
    # Trade metrics
    profitable_trades = [t for t in trades if t.get('pnl', 0) > 0]
    win_rate = len(profitable_trades) / len(trades) if trades else 0
    avg_profit = np.mean([t['pnl_pct'] for t in profitable_trades]) if profitable_trades else 0
    avg_loss = np.mean([t['pnl_pct'] for t in trades if t.get('pnl', 0) <= 0]) if trades else 0
    profit_factor = abs(sum(t['pnl'] for t in profitable_trades) / 
                       sum(t['pnl'] for t in trades if t.get('pnl', 0) <= 0)) if trades else 0
    
    return {
        'Total Return': total_return,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Max Drawdown': max_drawdown,
        'Win Rate': win_rate,
        'Avg Profit': avg_profit,
        'Avg Loss': avg_loss,
        'Profit Factor': profit_factor
    }

In [ ]:
def plot_strategy_analysis(df, symbol):
    """Enhanced visualization with multiple subplots"""
    fig, axes = plt.subplots(4, 1, figsize=(15, 20), gridspec_kw={'height_ratios': [2, 1, 1, 1]})
    
    # Price and Portfolio Plot
    axes[0].plot(df.index, df['Close'], label='Price', alpha=0.7)
    axes[0].plot(df.index, df['BB_High'], '--', label='BB High', alpha=0.5)
    axes[0].plot(df.index, df['BB_Low'], '--', label='BB Low', alpha=0.5)
    axes[0].scatter(df[df['Signal'] == 1].index, df[df['Signal'] == 1]['Close'], 
                   marker='^', color='g', label='Buy', alpha=1)
    axes[0].scatter(df[df['Signal'] == -1].index, df[df['Signal'] == -1]['Close'], 
                   marker='v', color='r', label='Sell', alpha=1)
    axes[0].set_title(f'{symbol} Price Action and Signals')
    axes[0].legend()
    
    # RSI and MFI
    axes[1].plot(df.index, df['RSI'], label='RSI', color='blue')
    axes[1].plot(df.index, df['MFI'], label='MFI', color='orange', alpha=0.7)
    axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5)
    axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5)
    axes[1].set_title('RSI and MFI')
    axes[1].legend()
    
    # MACD
    axes[2].plot(df.index, df['MACD'], label='MACD')
    axes[2].plot(df.index, df['MACD_Signal'], label='Signal')
    axes[2].bar(df.index, df['MACD'] - df['MACD_Signal'], alpha=0.3, label='Histogram')
    axes[2].set_title('MACD')
    axes[2].legend()
    
    # Portfolio Performance
    portfolio_returns = df['Portfolio'].pct_change()
    cumulative_returns = (1 + portfolio_returns).cumprod()
    axes[3].plot(df.index, cumulative_returns, label='Strategy Returns')
    axes[3].plot(df.index, (1 + df['Close'].pct_change()).cumprod(), 
                 label='Buy & Hold Returns', alpha=0.7)
    axes[3].set_title('Cumulative Returns Comparison')
    axes[3].legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Run the enhanced strategy
symbol = "TSLA"
df = prepare_data(symbol, "2020-01-01", "2025-05-01")
df = calculate_signals(df)
df, trades = backtest_strategy(df)
metrics = calculate_metrics(df, trades)

# Display results
print("\nStrategy Performance Metrics:")
for metric, value in metrics.items():
    if metric in ['Total Return', 'Max Drawdown', 'Win Rate', 'Avg Profit', 'Avg Loss']:
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value:.2f}")

# Plot the strategy analysis
plot_strategy_analysis(df, symbol)